# Creator production

Fig. 1(c). Match AIGC-adopting creators to non-adopting creators, then compare video volume, total valid views, and total full views.

### 1. The two arms

Focal is the treated arm. Comparison is the other arm. Nothing is paired yet.

In [1]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

from core.pipeline import ensure_synthetic, load_config

cfg = load_config(root)
spec = cfg["tasks"]["creator"]
pool = ensure_synthetic(root, cfg, rebuild=False)[{"creator": "creators", "consumer": "consumers", "video": "videos", "algorithm": "algorithm"}["creator"]]
focal = pool[pool["treated"] == 1].reset_index(drop=True)
comparison = pool[pool["treated"] == 0].reset_index(drop=True)
import pandas as pd
pd.DataFrame([{
    "focal_label": spec["focal"],
    "comparison_label": spec["comparison"],
    "n_focal": len(focal),
    "n_comparison": len(comparison),
    "exact": ", ".join(spec["exact"]),
    "caliper": ", ".join(f"{name} <= {limit}" for name, limit in (spec.get("caliper") or {}).items()) or "none",
}])


,focal_label,comparison_label,n_focal,n_comparison,exact,caliper
0,AIGC-adopting,non-adopting,1200,1200,"category, city, brand_tier",none


### 2. Exact cells

Units are blocked on the categorical covariates first. A focal unit whose cell has no comparison unit cannot be matched, before any distance is computed.

In [2]:
from core.matching import exact_cell_counts

cells = exact_cell_counts(focal, comparison, spec["exact"])
cells.sort_values(["n_comparison", "n_focal"], ascending=[True, False])


,category,city,brand_tier,n_focal,n_comparison
8,travel,east,tier_3,34,19
4,travel,north,tier_2,32,22
25,leisure,east,tier_2,30,22
7,travel,east,tier_2,32,24
30,food,north,tier_1,31,25
27,food,south,tier_1,35,26
2,travel,south,tier_3,34,27
11,beauty,south,tier_3,33,27
34,food,east,tier_2,41,29
3,travel,north,tier_1,45,30


### 3. Scale of the continuous covariates

Nearest neighbor uses a pooled z-score. The mean and standard deviation are taken over both arms together.

In [3]:
from core.matching import covariate_scale

covariate_scale(focal, comparison, spec["numeric"])


,covariate,pooled_mean,pooled_sd
0,uploads_14d,3.144609,1.135376
1,uploads_60d,7.831906,2.842064
2,plays_14d,98.862106,47.383632
3,plays_60d,300.265436,138.884936
4,watch_time_14d,195.107079,81.594532
5,watch_time_60d,587.076226,246.067027
6,followers,918.080430,853.665022


### 4. Nearest neighbor inside the cell

Each focal unit takes the comparison unit in the same cell with the smallest Euclidean distance. A caliper, when there is one, sets the distance to infinity and that neighbor is dropped. The ledger counts, per cell, how many focal units were kept and how many were dropped.

In [4]:
from core.matching import match_process

pairs, ledger = match_process(
    focal,
    comparison,
    spec["exact"],
    spec["numeric"],
    spec["outcomes"],
    spec.get("caliper") or {},
)
ledger.sort_values(["n_dropped", "n_focal"], ascending=[False, False])


,category,city,brand_tier,n_focal,n_comparison,n_matched,n_dropped,median_distance
3,travel,north,tier_1,45,30,45,0,1.541303
29,food,south,tier_3,42,38,42,0,1.513577
34,food,east,tier_2,41,29,41,0,1.772244
24,leisure,east,tier_1,40,34,40,0,1.669527
13,beauty,north,tier_2,39,32,39,0,1.383733
35,food,east,tier_3,38,34,38,0,1.732613
1,travel,south,tier_2,37,32,37,0,1.582768
21,leisure,north,tier_1,37,36,37,0,1.494133
0,travel,south,tier_1,35,37,35,0,1.750298
27,food,south,tier_1,35,26,35,0,1.988734


### 5. What was dropped, and how often a comparison unit is reused

Replacement means one comparison unit may be the neighbor for more than one focal unit.

In [5]:
import pandas as pd

reuse = pairs["comparison_row"].value_counts()
pd.DataFrame([{
    "focal_units": len(focal),
    "comparison_units": len(comparison),
    "exact_cells": len(ledger),
    "dropped_no_comparison_in_cell": int(ledger.loc[ledger["n_comparison"] == 0, "n_focal"].sum()),
    "dropped_by_caliper_or_empty_neighborhood": int(ledger.loc[ledger["n_comparison"] > 0, "n_dropped"].sum()),
    "pairs": len(pairs),
    "comparison_units_used": int(reuse.shape[0]),
    "max_reuse": int(reuse.max()) if len(reuse) else 0,
}])


,focal_units,comparison_units,exact_cells,dropped_no_comparison_in_cell,dropped_by_caliper_or_empty_neighborhood,pairs,comparison_units_used,max_reuse
0,1200,1200,36,0,0,1200,677,11


### 6. Eight pairs

Exact covariates are copied onto both sides. They agree by construction. `distance` is the z-score Euclidean distance of the chosen neighbor.

In [6]:
show = ["distance"]
for column in spec["exact"]:
    show += [f"{column}_focal", f"{column}_comparison"]
for column in spec.get("caliper") or {}:
    gap = f"{column}_gap"
    pairs[gap] = (pairs[f"{column}_focal"] - pairs[f"{column}_comparison"]).abs()
    show.append(gap)
pairs[show].head(8)


,distance,category_focal,category_comparison,city_focal,city_comparison,brand_tier_focal,brand_tier_comparison
0,1.457673,travel,travel,south,south,tier_1,tier_1
1,0.989412,travel,travel,south,south,tier_1,tier_1
2,1.774577,travel,travel,south,south,tier_1,tier_1
3,1.847817,travel,travel,south,south,tier_1,tier_1
4,1.211666,travel,travel,south,south,tier_1,tier_1
5,1.800166,travel,travel,south,south,tier_1,tier_1
6,2.110708,travel,travel,south,south,tier_1,tier_1
7,2.070506,travel,travel,south,south,tier_1,tier_1


### 7. Balance before and after

Standardized mean difference of each covariate, on the raw arms and on the matched pairs.

In [7]:
from core.balance import balance_before_after

balance = balance_before_after(focal, comparison, pairs, list(spec["exact"]) + list(spec["numeric"]))
balance


,covariate,mean_focal_before,mean_comparison_before,standardized_difference_before,mean_focal_after,mean_comparison_after,standardized_difference_after
0,followers,942.667500,893.493361,0.057603,942.667500,820.969834,0.153943
1,plays_14d,97.997251,99.726962,-0.036495,97.997251,94.261575,0.087362
2,plays_60d,299.140150,301.390721,-0.016198,299.140150,285.437389,0.109196
3,uploads_14d,3.124588,3.164630,-0.035258,3.124588,3.072618,0.049964
4,uploads_60d,7.878604,7.785207,0.032853,7.878604,7.566407,0.119314
5,watch_time_14d,193.873975,196.340184,-0.030216,193.873975,189.065377,0.065976
6,watch_time_60d,582.602695,591.549757,-0.036351,582.602695,572.399588,0.045925


### 8. The contrast

Mean of focal minus comparison. The standard error is Abadie-Imbens, with up to five within-arm neighbors. The Hodges-Lehmann median and the Wilcoxon p-value are the rank summary beside that mean.

In [8]:
from core.imbens import contrast_table

effects = contrast_table(
    focal,
    comparison,
    pairs,
    spec["outcomes"],
    spec["exact"],
    spec["numeric"],
    n_neighbors=int(cfg.get("imbens_neighbors", 5)),
)
out = root / "data" / "results"
out.mkdir(parents=True, exist_ok=True)
pairs.to_csv(out / "creator_pairs.csv", index=False)
effects.to_csv(out / "creator_effects.csv", index=False)
balance.to_csv(out / "creator_balance.csv", index=False)
effects


,outcome,n_pairs,hl_focal_minus_comparison,mean_difference,wilcoxon_statistic,wilcoxon_p,variance,standard_error,z_statistic,p_value,ci95_low,ci95_high
0,video_volume,1200,13.920069,13.924147,0.0,8.134563e-198,0.025717,0.160365,86.827884,0.000000,13.609831,14.238462
1,total_valid_views,1200,9.172297,9.090333,316273.0,2.457727e-04,11.662985,3.415111,2.661797,0.007772,2.396715,15.783952
2,total_full_views,1200,0.553059,0.598251,351238.0,4.504320e-01,0.818402,0.904656,0.661302,0.508419,-1.174875,2.371376
